# BiS Guild — read a raid log, no PowerShell

**What this does:** reads your WoW combat log and prints a block of text. You paste that text into BiS Guild in game (`/bisg paste`), and the raid is on your guild's record — attendance, consumables, deaths, boss kills, gear.

**Is it safe?** It runs on **Google's computer, not yours**. It only reads the log you give it, and nothing here can touch your PC. The log goes to your own Google session and nowhere else.

**Where this comes from:** we publish BiS Guild in two places only — its CurseForge page and **github.com/blankfer/BiSGuild**. If you were sent this notebook from anywhere else, it isn't ours.

**How:**
1. Press **Runtime → Run all** (or the ▶ on each step below).
2. Google shows *"This notebook was not authored by Google"* — that's normal for any notebook not made by Google. Press **Run anyway**.
3. In step 3, choose your log: `World of Warcraft\_classic_beta_\Logs\WoWCombatLog-....txt` (zipping it first makes the upload much faster).
4. Press **Copy** under the text, then in game type `/bisg paste`, paste with **Ctrl+V**, and press **Import**.

In [ ]:
# Step 2 - get the log reader: the NEWEST RELEASED BiS Guild, straight from its GitHub page.
# Nothing is copied into this notebook, so the notebook never goes stale: every release is used as it ships.
import json, os, re, sys, urllib.request

REPO = "blankfer/BiSGuild"
HERE = "/content/bisguild"
os.makedirs(HERE, exist_ok=True)

def newest_release():
    tags = json.load(urllib.request.urlopen("https://api.github.com/repos/%s/tags?per_page=100" % REPO))
    vers = []
    for t in tags:
        m = re.match(r"^v(\d+)\.(\d+)\.(\d+)$", t["name"])
        if m:
            vers.append((tuple(int(x) for x in m.groups()), t["name"]))
    return max(vers)[1]

source = os.environ.get("BISGUILD_SOURCE")       # for testing a copy on disk; leave unset
tag = None
if source:
    for name in ("logreport.py", "consumables.txt", "gearcheck.txt"):
        with open(os.path.join(source, name), "rb") as src, open(os.path.join(HERE, name), "wb") as dst:
            dst.write(src.read())
    print("using the reader from", source)
else:
    tag = newest_release()
    for name in ("logreport.py", "consumables.txt", "gearcheck.txt"):
        url = "https://raw.githubusercontent.com/%s/%s/dev/%s" % (REPO, tag, name)
        urllib.request.urlretrieve(url, os.path.join(HERE, name))
    print("BiS Guild", tag, "- reader downloaded from github.com/" + REPO)

sys.path.insert(0, HERE)
if "logreport" in sys.modules:
    del sys.modules["logreport"]
import logreport
if not hasattr(logreport, "paste_text"):
    raise SystemExit("This release (%s) cannot make a paste yet - BiS Guild 0.2.0 or newer is needed." % tag)
print("ready: paste layout", logreport.PASTE_VERSION)

In [ ]:
# Step 3 - your combat log. Zipped is fine (and much faster to upload).
import glob, os, shutil, zipfile
LOGS = "/content/Logs"
shutil.rmtree(LOGS, ignore_errors=True)
shutil.rmtree("/content/WTF", ignore_errors=True)        # a fresh run: only the logs chosen now
os.makedirs(LOGS)

uploaded = {}
if os.environ.get("BISGUILD_UPLOAD"):                    # for testing; leave unset
    p = os.environ["BISGUILD_UPLOAD"]
    uploaded[os.path.basename(p)] = open(p, "rb").read()
else:
    from google.colab import files
    uploaded = files.upload()

for name, data in uploaded.items():
    path = os.path.join("/content", name)
    with open(path, "wb") as fh:
        fh.write(data)
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            for member in z.namelist():
                if member.lower().endswith(".txt"):
                    target = os.path.join(LOGS, os.path.basename(member))
                    with z.open(member) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
    elif name.lower().endswith(".txt"):
        shutil.move(path, os.path.join(LOGS, name))
logs = sorted(glob.glob(os.path.join(LOGS, "*.txt")))
print(len(logs), "log(s):", ", ".join(os.path.basename(l) for l in logs) or "none - choose a WoWCombatLog .txt or .zip")

In [ ]:
# Step 4 - read them, and print the text to paste into BiS Guild (/bisg paste in game).
import contextlib, html, io
for path in logs:
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        rc = logreport.main(path)
    first = out.getvalue().splitlines()[:1]
    print(os.path.basename(path), "-", first[0] if first else "read")

if not logs:
    raise SystemExit("no log was chosen in step 3")
nightly, gear = logreport.build_record(logreport.read_history(logs[-1]))
raids = sorted(k for k, n in nightly.items() if n.get("v") == 2)
if not raids:
    raise SystemExit("no raid boss kills in these logs - a dungeon boss is not a raid night")
for key in raids:
    n = nightly[key]
    print("  raid", key, n.get("zone") or "", "-", n["kills"], "kill(s),", len(n["raiders"]), "raider(s)")
text = logreport.paste_text(nightly, gear)

from IPython.display import HTML, display
display(HTML("""
<p><b>Copy this, then in game: <code>/bisg paste</code>, Ctrl+V, Import.</b></p>
<button onclick="navigator.clipboard.writeText(document.getElementById('bisg').value);this.innerText='Copied - now /bisg paste in game'"
        style="font-size:16px;padding:6px 18px">Copy</button>
<textarea id="bisg" readonly style="width:100%%;height:180px;font-family:monospace;font-size:11px">%s</textarea>
""" % html.escape(text)))